# Multi-Model Training Pipeline with Explainability

This notebook demonstrates how to use the high-level `run_multi_model_pipeline` wrapper. It will automatically train and compare multiple algorithms, save the best one to disk, and allow us to visualize its performance.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
from sklearn.datasets import make_classification
from src.pipeline import run_multi_model_pipeline
from src.utils import plot_feature_importance, plot_confusion_matrix, load_model

### 1. Generate / Load Data

In [ ]:
# Generate a mock tabular dataset
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15, n_classes=2, random_state=42)
# Give features some arbitrary names for better visualization
feature_names = [f'Feature_{i}' for i in range(20)]
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

# Save it to data directory as an example
os.makedirs('../data', exist_ok=True)
df.to_csv('../data/sample_data.csv', index=False)

# Load it back
df = pd.read_csv('../data/sample_data.csv')
df.head()

### 2. Run Pipeline (Train, Tune, Evaluate, and Auto-Save)

In [ ]:
# Run the pipeline and evaluate configured models in src/config.py
# This will automatically save the best model to ../models/best_model.joblib
results = run_multi_model_pipeline(df, target_col='target')

print("\n--- Model Comparison ---")
display(results['comparison_df'])

print(f"\nBest Overall Model: {results['best_model_name']}")
print(f"Best Parameters: {results['best_params']}")

### 3. Model Explainability and Diagnostics

In [ ]:
# Plot the Confusion Matrix to see where the model makes mistakes
plot_confusion_matrix(results['y_test'], results['best_y_pred'], labels=[0, 1])

In [ ]:
# Plot Feature Importances to understand what drives the model's decisions
plot_feature_importance(results['best_model'], results['feature_names'])

### 4. Load the Model for Inference Later
You can load your saved `.joblib` model at any time (e.g., in an API script) without retraining it.

In [ ]:
loaded_model = load_model('../models/best_model.joblib')
print("Model loaded successfully:", loaded_model)